# Week 8 (starter): Integrated LLM System (capstone)

A small skeleton that integrates three techniques you already built: retrieval (Week 5), a tool with routing (Week 4), and a grounded prompt (Week 3). Orchestration runs locally with no key; synthesis and the LLM-as-judge are guarded. **Reuse your own components**; this is a starting frame, not the finished system. Cells marked **TODO (you)** are yours.

In [1]:
#%pip install faiss-cpu
# Gemini import
from openai import OpenAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [2]:
#Week 5 corpus, retrieval and chunking system
import os, re, pathlib, requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
!pip install -q faiss-cpu
import faiss

os.environ["HF_HOME"] = str((pathlib.Path(".") / ".hf_cache").resolve())
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

DOC_URLS = {
    'astar_32u4': 'https://www.pololu.com/docs/0J61/all',
    'astar_robot_controller': 'https://www.pololu.com/docs/0J66/all',
    'zumo_32u4': 'https://www.pololu.com/docs/0J63/all',
    'zumo_shield': 'https://www.pololu.com/docs/0J57',
    'three_pi_plus': 'https://www.pololu.com/docs/0J83/all',
    'zumo_chassis': 'https://www.pololu.com/docs/0J54/all',
    'qtr_sensors': 'https://www.pololu.com/docs/0J19/all',
    'mc33926_motor_shield': 'https://www.pololu.com/docs/0j55/all'
}

def load_doc(url):
    response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    chunks = []
    for element in soup.select("h1, h2, h3, h4, p, li"):
        text = re.sub(r"\s+", " ", element.get_text(" ", strip=True)).strip()
        if text:
            chunks.append(text)
    return "\n\n".join(chunks)

DOCS = {}
for name, url in DOC_URLS.items():
    try:
        DOCS[name] = load_doc(url)
        print(f"{name:22s} -> {len(DOCS[name]):,} characters")
    except requests.RequestException as e:
        print(f"{name:22s} -> FAILED: {e}")

def chunk_fixed(text, size, overlap):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += size - overlap

    return chunks

def chunk_paragraph(text):
    return [re.sub(r"\s+", " ", paragraph).strip()
        for paragraph in text.split("\n\n")
        if paragraph.strip()]

configs = {
    "small_120_20": {name: chunk_fixed(text, 120, 20)
        for name, text in DOCS.items()},
    "large_320_40": {name: chunk_fixed(text, 320, 40)
        for name, text in DOCS.items()},
    "paragraph": {name: chunk_paragraph(text)
        for name, text in DOCS.items()}
}

for config_name, documents in configs.items():
    total = sum(len(chunks) for chunks in documents.values())
    print(f"{config_name:16s} -> {total:,} chunks")
    for name, chunks in documents.items():
        print(f"    {name:25s}: {len(chunks)}")


# Prepare each chunking configuration for FAISS retrieval.
# Each chunk keeps data identifying which document it came from.

prepared = {}

for config_name, documents in configs.items():
    all_chunks = []
    metadata = []

    for document_name, chunks in documents.items():
        for chunk_number, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            metadata.append({
                "document": document_name,
                "chunk_number": chunk_number
            })

    prepared[config_name] = {
        "chunks": all_chunks,
        "metadata": metadata
    }

    print(
        f"{config_name:16s} -> "
        f"{len(all_chunks):,} chunks"
    )


def build_index(chunks):
    embeddings = embedder.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index


# Build one FAISS index for each chunking configuration.
indices = {}

for config_name, data in prepared.items():
    indices[config_name] = build_index(data["chunks"])

    print(
        f"{config_name}: "
        f"FAISS index contains {indices[config_name].ntotal} vectors"
    )


def retrieve(config_name, query, k=3):
    data = prepared[config_name]
    index = indices[config_name]

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, positions = index.search(query_embedding, k)

    results = []

    for score, position in zip(scores[0], positions[0]):
        if position >= 0:
            results.append({
                "score": float(score),
                "text": data["chunks"][position],
                "document": data["metadata"][position]["document"],
                "chunk_number": data["metadata"][position]["chunk_number"]
            })

    return results


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

astar_32u4             -> 81,327 characters
astar_robot_controller -> 69,057 characters
zumo_32u4              -> 101,830 characters
zumo_shield            -> 1,216 characters
three_pi_plus          -> 91,048 characters
zumo_chassis           -> 8,518 characters
qtr_sensors            -> 26,947 characters
mc33926_motor_shield   -> 32,927 characters
small_120_20     -> 4,104 chunks
    astar_32u4               : 809
    astar_robot_controller   : 687
    zumo_32u4                : 1012
    zumo_shield              : 12
    three_pi_plus            : 905
    zumo_chassis             : 84
    qtr_sensors              : 268
    mc33926_motor_shield     : 327
large_320_40     -> 1,468 chunks
    astar_32u4               : 289
    astar_robot_controller   : 246
    zumo_32u4                : 362
    zumo_shield              : 4
    three_pi_plus            : 324
    zumo_chassis             : 30
    qtr_sensors              : 96
    mc33926_motor_shield     : 117
paragraph        -> 2,650 ch

Batches:   0%|          | 0/129 [00:00<?, ?it/s]

small_120_20: FAISS index contains 4104 vectors


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

large_320_40: FAISS index contains 1468 vectors


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

paragraph: FAISS index contains 2650 vectors


In [3]:
#Week 4 tools - adapted to fit with the corpus used instead of the support ticket format that I used on week 4

import json

TOOLS = [
    {"name": "search_docs",
     "description": "Search all Pololu robotics documentation for technical information.",
     "parameters": {"query": "string", "config_name": "string", "k": "integer"}},
    {"name": "get_document_info",
     "description": "Search one specific Pololu document for relevant passages.",
     "parameters": {"document": "string", "query": "string", "config_name": "string"}}
]

class ToolArgError(Exception):
    pass

def validate(name, args):
    if name not in {"search_docs", "get_document_info"}:
        raise ToolArgError(f"Unknown tool: {name}")
    if not isinstance(args, dict):
        raise ToolArgError("Arguments must be a dictionary")
    required = ({"query", "config_name", "k"} if name == "search_docs"
                else {"document", "query", "config_name"})
    if set(args) != required:
        raise ToolArgError(f"Expected exactly these fields: {required}")
    if not isinstance(args["query"], str) or not args["query"].strip():
        raise ToolArgError("query must be a non-empty string")
    if args["config_name"] not in configs:
        raise ToolArgError("Invalid chunk configuration")
    if name == "search_docs":
        k = args["k"]
        if type(k) is not int or not 1 <= k <= 10:
            raise ToolArgError("k must be an integer between 1 and 10")
    elif args["document"] not in DOCS:
        raise ToolArgError("Unknown document")

def search_docs(query, config_name, k):
    return retrieve(config_name, query, k)

def get_document_info(document, query, config_name):
    data = prepared[config_name]
    candidates = [
        (i, chunk) for i, chunk in enumerate(data["chunks"])
        if data["metadata"][i]["document"] == document
    ]
    if not candidates:
        return []
    texts = [chunk for _, chunk in candidates]
    qvec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    vectors = embedder.encode(texts, normalize_embeddings=True).astype("float32")
    scores = (vectors @ qvec.T).flatten()
    ranked = scores.argsort()[::-1][:3]
    return [
        {"document": document,
         "chunk_number": data["metadata"][candidates[i][0]]["chunk_number"],
         "text": texts[i], "score": float(scores[i])}
        for i in ranked
    ]

IMPL = {"search_docs": search_docs, "get_document_info": get_document_info}

def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)
        return {"ok": True, "tool": name, "output": output}
    except ToolArgError as e:
        return {"ok": False, "tool": name, "error_type": "invalid_arguments", "message": str(e)}
    except Exception as e:
        return {"ok": False, "tool": name, "error_type": "execution_error", "message": str(e)}

print("Week 4 documentation tools loaded")


Week 4 documentation tools loaded


In [4]:
#Week 7 - using dynamic few shot examples, I adapted the dynamic fewshot example retrieval system from week7 and integrated it into our pipeline using the week 5 corpus
FEW_SHOT_EXAMPLES = [
    {"q": "What sensors does the Zumo use for line detection?",
        "answer": "Search the QTR sensor and Zumo documentation. Explain line detection using the retrieved evidence and identify the relevant sources."},
    {"q": "Which controller is used by the A-Star 32U4?",
        "answer": "Search the A-Star 32U4 documentation and identify the controller using the retrieved passages."},
    {"q": "How do QTR reflectance sensors work?",
        "answer": "Retrieve relevant QTR sensor documentation and summarize how the sensors detect reflected light, using only supported technical details."},
    {"q": "Where can I find information about the motor driver shield?",
        "answer": "Search the MC33926 motor shield documentation and summarize the relevant information from the retrieved passages."},
    {"q": "What is the difference between the Zumo chassis and Zumo 32U4?",
        "answer": "Retrieve documentation about both products, compare only the details supported by the passages, and cite the relevant document names."},
    {"q": "How do I program the A-Star robot controller?",
        "answer": "Search the A-Star robot controller documentation and summarize the documented programming steps. State when the retrieved evidence is insufficient."}
]

# Embedding the examples
few_shot_embeddings = embedder.encode(
    [e["q"] for e in FEW_SHOT_EXAMPLES],
    normalize_embeddings=True
).astype("float32")


def dynamic_few_shot(query, k=2):
    q_vec = embedder.encode(
        [query], normalize_embeddings=True
    ).astype("float32")
    scores = (few_shot_embeddings @ q_vec.T).flatten()
    ranked = scores.argsort()[::-1][:k]
    return [FEW_SHOT_EXAMPLES[i] for i in ranked]


# Integrate few-shot examples into the existing assistant prompt.
def assistant(q, config_name="paragraph", k=3):
    path = route(q)
    docs = retrieve(config_name, q, k=k)
    examples = dynamic_few_shot(q)

    context = "\n".join(
        f"- [{r['document']}] {r['text']}" for r in docs
    )
    shots = "\n".join(
        f"Question: {e['q']}\nAnswer approach: {e['answer']}"
        for e in examples
    )

    prompt = (
        "You are a robotics documentation assistant. "
        "Answer using retrieved evidence and tool results only. "
        "If evidence is insufficient, say so. Do not treat examples as facts "
        "about the current question.\n\n"
        f"Relevant examples:\n{shots}\n\n"
        f"Retrieved documentation:\n{context}\n\n"
        f"Question: {q}\nAnswer:"
    )

    return {
        "route": path,
        "retrieved_docs": docs,
        "few_shot_examples": examples,
        "prompt": prompt
    }



#part2

## Part 2: Evaluate (model-free metrics + guarded judge)
Routing accuracy and answer-grounding are measurable without the model. The LLM-as-judge pass is guarded.

In [5]:
# WEEK 8 — Integrated system evaluation of Weeks 4, 5, and 7
import os, re, json, statistics
from pathlib import Path
from openai import OpenAI

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
CONFIG, K, MODEL = "paragraph", 3, "gemini-3.5-flash-lite"
API_LIMIT, api_calls = 10, 0

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY is missing.")

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    timeout=120, max_retries=0
)

def call_gemini(prompt):
    global api_calls
    if api_calls >= API_LIMIT:
        raise RuntimeError("Gemini API call limit reached.")
    api_calls += 1
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content or ""

def parse_json_response(raw):
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())
    return json.loads(raw)

# 10 held-out robotics documentation questions.
EVAL_SET = [
    {"q": "What controller is used by the A-Star 32U4?",
     "tool": "get_document_info", "doc": "astar_32u4"},
    {"q": "How do I program the A-Star robot controller?",
     "tool": "get_document_info", "doc": "astar_robot_controller"},
    {"q": "How do QTR sensors detect reflected light?",
     "tool": "get_document_info", "doc": "qtr_sensors"},
    {"q": "What features does the Zumo 32U4 have?",
     "tool": "get_document_info", "doc": "zumo_32u4"},
    {"q": "What does the Zumo chassis documentation explain?",
     "tool": "get_document_info", "doc": "zumo_chassis"},
    {"q": "How does the MC33926 motor shield work?",
     "tool": "get_document_info", "doc": "mc33926_motor_shield"},
    {"q": "What is the 3pi+ robot?",
     "tool": "get_document_info", "doc": "three_pi_plus"},
    {"q": "How do I use the Zumo shield?",
     "tool": "get_document_info", "doc": "zumo_shield"},
    {"q": "Which documents discuss line sensing?",
     "tool": "search_docs", "doc": "qtr_sensors"},
    {"q": "Find information about controlling robot motors.",
     "tool": "search_docs", "doc": "mc33926_motor_shield"}
]

def run_local_case(item):
    # Explicit tool selection
    if item["tool"] == "get_document_info":
        args = {"document": item["doc"], "query": item["q"], "config_name": CONFIG}
    else:
        args = {"query": item["q"], "config_name": CONFIG, "k": K}

    result = dispatch(item["tool"], args)
    docs = result["output"] if result["ok"] else []

    # Independent Week 5 retrieval test.
    global_result = dispatch("search_docs", {
        "query": item["q"], "config_name": CONFIG, "k": K
    })
    global_docs = global_result["output"] if global_result["ok"] else []
    hit = item["doc"] in [d["document"] for d in global_docs]

    # Week 7 dynamic few-shot retrieval.
    examples = dynamic_few_shot(item["q"], k=2)
    shots = "\n".join(
        f"Question: {e['q']}\nApproach: {e['answer']}" for e in examples
    )
    evidence = "\n".join(
        f"[{d['document']}] {d['text']}" for d in docs
    ) or "No evidence retrieved."

    prompt = f"""You are a Pololu robotics documentation assistant.
Answer concisely using only retrieved evidence.
Examples are guidance, not factual evidence.
Do not invent technical details. If evidence is insufficient, say so.
Identify source documents when possible.

Relevant examples:
{shots}

Retrieved evidence:
{evidence}

Question: {item['q']}
Answer:"""

    return {
        "question": item["q"], "tool": item["tool"],
        "expected_document": item["doc"],
        "retrieved_documents": [d["document"] for d in docs],
        "global_top_k_documents": [d["document"] for d in global_docs],
        "retrieval_hit_at_3": hit,
        "evidence": [{"document": d["document"], "text": d["text"]} for d in docs],
        "examples": [e["q"] for e in examples],
        "prompt": prompt, "answer": None
    }

records = [run_local_case(item) for item in EVAL_SET]

# Five answer requests, two cases per request.
for start in range(0, len(records), 2):
    batch = records[start:start + 2]
    payload = [{"id": start + i, "prompt": r["prompt"]}
               for i, r in enumerate(batch)]
    try:
        raw = call_gemini(
            "Answer each case independently using its prompt. Return valid "
            "JSON only: {\"answers\":[{\"id\":0,\"answer\":\"...\"}]}. "
            "Include every ID.\n" + json.dumps(payload)
        )
        outputs = parse_json_response(raw)["answers"]
        by_id = {x["id"]: x["answer"] for x in outputs}
        for i, r in enumerate(batch):
            r["answer"] = by_id.get(start + i)
    except Exception as e:
        for r in batch:
            r["answer_error"] = str(e)

# One batched LLM-as-judge request.
judge_input = [
    {"id": i, "question": r["question"], "evidence": r["evidence"],
     "answer": r["answer"] or "[MISSING ANSWER]"}
    for i, r in enumerate(records)
]
judge_results, judge_error = [], None

try:
    raw = call_gemini(
        "Score each answer from 1 to 5 on correctness, grounding in the "
        "supplied evidence, and relevance. Missing answers score 1 on all "
        "dimensions. Return JSON only: {\"scores\":[{\"id\":0,"
        "\"correctness\":1,\"grounding\":1,\"relevance\":1}]}. "
        "Include all IDs exactly once.\n" + json.dumps(judge_input)
    )
    judge_results = parse_json_response(raw)["scores"]
    assert len(judge_results) == len(records)
    assert {s["id"] for s in judge_results} == set(range(len(records)))
    for s in judge_results:
        for dim in ("correctness", "grounding", "relevance"):
            if type(s.get(dim)) is not int or not 1 <= s[dim] <= 5:
                raise ValueError(f"Invalid judge score: {s}")
except Exception as e:
    judge_error = str(e)
    print("Judge failed; no automatic retry:", judge_error)

# Model-free retrieval metric and judge means.
retrieval_hit_rate = sum(r["retrieval_hit_at_3"] for r in records) / len(records)
judge_means = (
    {dim: round(statistics.mean(s[dim] for s in judge_results), 2)
     for dim in ("correctness", "grounding", "relevance")}
    if judge_results else None
)

summary = {
    "evaluation_cases": len(records),
    "retrieval_hit_rate_at_3": round(retrieval_hit_rate, 3),
    "judge_cases": len(judge_results),
    "judge_mean_scores_out_of_5": judge_means,
    "api_calls_used": api_calls,
    "api_call_limit": API_LIMIT,
    "judge_error": judge_error
}

for i, r in enumerate(records):
    r.pop("prompt", None)
    if i < len(judge_results):
        r["judge"] = next((s for s in judge_results if s["id"] == i), None)

(RESULTS / "capstone_evaluation.json").write_text(
    json.dumps({"summary": summary, "results": records}, indent=2),
    encoding="utf-8"
)
print(json.dumps(summary, indent=2))
assert api_calls <= API_LIMIT



{
  "evaluation_cases": 10,
  "retrieval_hit_rate_at_3": 0.9,
  "judge_cases": 10,
  "judge_mean_scores_out_of_5": {
    "correctness": 5,
    "grounding": 5,
    "relevance": 5
  },
  "api_calls_used": 6,
  "api_call_limit": 10,
  "judge_error": null
}


In [6]:
# WEEK 8 — Investigate retrieval failure and test mitigation
import json
from pathlib import Path

failure_dir = Path("results")
failure_dir.mkdir(exist_ok=True)

# Find cases where the expected document was absent from global top-3.
failures = [r for r in records if not r.get("retrieval_hit_at_3", False)]

print(f"Retrieval failures: {len(failures)} / {len(records)}")

if not failures:
    print("No retrieval miss found. Inspect the lowest-scoring judge cases instead.")
else:
    for i, failure in enumerate(failures, 1):
        question = failure["question"]
        target = failure["expected_document"]

        print(f"\nFailure {i}")
        print("Question:", question)
        print("Expected document:", target)
        print("Retrieved documents:", failure["global_top_k_documents"])

        # Compare all existing Week 5 chunking strategies.
        comparison = {}
        for config_name in configs:
            results = retrieve(config_name, question, k=3)
            found = target in [r["document"] for r in results]
            comparison[config_name] = {
                "target_found": found,
                "results": [
                    {"document": r["document"], "score": r["score"],
                     "chunk_number": r["chunk_number"], "text": r["text"][:300]}
                    for r in results
                ]
            }
            print(f"{config_name}: target found = {found}")
            for r in results:
                print(f"  {r['document']} | score={r['score']:.3f}")

        # Mitigation: search the known document directly.
        fallback = dispatch("get_document_info", {
            "document": target,
            "query": question,
            "config_name": CONFIG
        })

        fallback_results = fallback.get("output", []) if fallback["ok"] else []
        mitigation_worked = bool(fallback_results)

        print("Targeted-document fallback succeeded:", mitigation_worked)
        for result in fallback_results[:2]:
            print("  Evidence:", result["text"][:300])

        report = {
            "question": question,
            "expected_document": target,
            "original_retrieval_hit": False,
            "chunking_comparison": comparison,
            "fallback_succeeded": mitigation_worked,
            "fallback_evidence": fallback_results[:2],
            "cause": "To be determined from the chunking comparison and retrieved passages."
        }

        (failure_dir / "failure_analysis.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )


Retrieval failures: 1 / 10

Failure 1
Question: Find information about controlling robot motors.
Expected document: mc33926_motor_shield
Retrieved documents: ['zumo_chassis', 'zumo_shield', 'zumo_32u4']
small_120_20: target found = False
  zumo_32u4 | score=0.612
  astar_robot_controller | score=0.593
  zumo_32u4 | score=0.587
large_320_40: target found = False
  astar_robot_controller | score=0.547
  three_pi_plus | score=0.538
  three_pi_plus | score=0.534
paragraph: target found = False
  zumo_chassis | score=0.642
  zumo_shield | score=0.642
  zumo_32u4 | score=0.642
Targeted-document fallback succeeded: True
  Evidence: Motor Control
  Evidence: Motors


In [7]:
"""
Failure Analysis: Retrieval Failure

One failure that occurred during testing was when the system was asked to find
information about controlling robot motors, but failed to retrieve the expected
MC33926 motor shield documentation within the top three results. This occurred
across all three chunking strategies, suggesting that the broad wording of the
question caused the retrieval system to rank other robotics documents more highly
than the intended document. To address this issue, the get_document_info tool was
used to search the specific document directly, which successfully retrieved relevant
passages about motor control. This demonstrates how targeted retrieval can help improve
the system's ability to find relevant information, although it still depends on
knowing which document to search.
"""

"\nFailure Analysis: Retrieval Failure\n\nOne failure that occurred during testing was when the system was asked to find \ninformation about controlling robot motors, but failed to retrieve the expected\nMC33926 motor shield documentation within the top three results. This occurred \nacross all three chunking strategies, suggesting that the broad wording of the \nquestion caused the retrieval system to rank other robotics documents more highly \nthan the intended document. To address this issue, the get_document_info tool was \nused to search the specific document directly, which successfully retrieved relevant \npassages about motor control. This demonstrates how targeted retrieval can help improve \nthe system's ability to find relevant information, although it still depends on \nknowing which document to search.\n"

## Part 3 and 4: failure, submit, curate
**TODO (you):** show one failure of the integrated system, trace it to a component, and mitigate. Write a README documenting the architecture and technique choices, then curate the repo as your portfolio piece. Rubric: integration by reuse (30), evaluation incl judge (25), failure traced (15), README (20), portfolio curation (10).